# K-Nearest Neighbors for weather risk 

You will build **three** `KNeighborsClassifier` models on the same tabular weather dataset:

1. **Wind KNN** — features for *wind-related* risk  
2. **Rain KNN** — features for *precipitation / moisture*  
3. **Storm KNN** — features for *severe* conditions (like lightning) 


**Primary API:** [`sklearn.neighbors.KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

## Learning roadmap (do these in order)

1. **Understand the CSV** — Open the file, decide what is a *feature* vs a *label*.
2. **Define targets** —  `unsafe_weather`/ `safe_weather`, you can train all three KNNs to predict that label using **different feature subsets** (wind-only, rain-only, storm-only). Each Knn should be differnt
3. **Clean types** — Convert labels to integers; coerce features to numeric; drop or encode non-numeric columns you need.
4. **Balance** — downsample/upsample or use class weights (KNN in sklearn does not support `class_weight`; balancing or choosing metrics carefully is important).
5. **Split data** — `train_test_split` with a fixed `random_state` for reproducibility.
6. **Scale features** — KNN is distance-based; fit `StandardScaler` **only on training data**, then transform train and test.
7. **Train three classifiers** — Instantiate `KNeighborsClassifier`, set `n_neighbors` (\(k\)), `fit` on scaled training data.
8. **Evaluate** — Accuracy, confusion matrix, precision/recall/F1 (especially for the minority “unsafe” class).
9. **Tune \(k\)** — Try a small grid of \(k\) values (odd numbers often avoid ties); compare validation or test performance.

## 0. Imports

Uncomment or add any extra imports you need.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

## 1. Configuration and loading data

**TODO:** Set `CSV_PATH` to balanced or raw dataset. Run `ml_dataset_scripts.py` from this folder first if you need to inspect counts; export a balanced CSV.

In [6]:
# Path relative to this notebook (usually backend/calculations)
CSV_PATH = Path("ml_ready_dataset (1).csv")
#print(Path("ml_ready_dataset (1).csv"))


#If the file is missing, create a tiny synthetic dataset 
if not CSV_PATH.is_file():
    print(f"Warning: {CSV_PATH} not found. Using synthetic data for pipeline practice.")
    rng = np.random.default_rng(42)
    n = 400
    df = pd.DataFrame(
        {
            "wind_speed": rng.lognormal(3, 0.5, n),
            "wind_gust": rng.lognormal(3.2, 0.5, n),
            "precip_inches": rng.exponential(0.1, n),
            "humidity_pct": rng.uniform(20, 100, n),
            "lightning_strikes_10mi": rng.poisson(2, n),
            "unsafe_weather": rng.choice([0, 1], n, p=[0.85, 0.15]),
        }
    )
else:

    df = pd.read_csv(CSV_PATH)


print(df.shape)
print(df.dtypes)
df.head()

(4475051, 23)
max_temp_f            float64
min_temp_f            float64
temp_range_f          float64
max_dewpoint_f        float64
min_dewpoint_f        float64
dewpoint_range_f      float64
precip_in             float64
avg_wind_speed_kts    float64
snow_in               float64
avg_feel              float64
icing_risk              int64
has_snow                int64
has_precip              int64
high_wind               int64
below_freezing          int64
MONTH                   int64
DEP_1hrpre_num        float64
DEP_1hrpost_num       float64
unsafe_weather          int64
FL_DATE                   str
ORIGIN                    str
DEST                      str
MKT_CARRIER               str
dtype: object


,max_temp_f,min_temp_f,temp_range_f,max_dewpoint_f,min_dewpoint_f,dewpoint_range_f,precip_in,avg_wind_speed_kts,snow_in,avg_feel,...,high_wind,below_freezing,MONTH,DEP_1hrpre_num,DEP_1hrpost_num,unsafe_weather,FL_DATE,ORIGIN,DEST,MKT_CARRIER
0,27.0,22.0,5.0,21.9,19.4,2.5,0.0200,3.128315,0.1000,19.545850,...,0,1,1,14.0,24.0,0,2023-01-02,MSP,SRQ,G4
1,45.0,34.0,11.0,45.0,32.0,13.0,0.4800,4.084188,0.0000,37.493580,...,0,0,1,26.0,9.0,0,2023-01-03,BOS,SRQ,G4
2,30.0,18.0,12.0,27.0,15.1,11.9,0.0100,8.429070,0.2000,14.716321,...,0,1,1,22.0,10.0,0,2023-01-05,MSP,SRQ,G4
3,43.0,28.0,15.0,30.0,21.9,8.1,0.0001,8.255275,0.0001,28.594500,...,0,1,1,18.0,22.0,0,2023-01-09,BOS,SRQ,G4
4,44.0,29.0,15.0,26.1,21.0,5.1,0.0000,7.125605,0.0000,27.086977,...,0,1,1,8.0,1.0,0,2023-01-09,IND,SRQ,G4


In [16]:
# Adding normalizing 
X = (df['max_temp_f'].to_numpy())
from sklearn.preprocessing import MinMaxScaler
data = df.drop(columns=['FL_DATE', 'ORIGIN', 'DEST', 'MKT_CARRIER'])
scaler = MinMaxScaler()
print(scaler.fit(data))
print(scaler.data_max_)
print(scaler.transform(data))


MinMaxScaler()
[119.       97.       53.       84.       78.1      61.       22.5
  26.67757  11.5     105.18889   1.        1.        1.        1.
   1.       11.      107.      107.        1.     ]
[[0.21367521 0.31818182 0.07692308 ... 0.13084112 0.22429907 0.        ]
 [0.36752137 0.42727273 0.19230769 ... 0.24299065 0.08411215 0.        ]
 [0.23931624 0.28181818 0.21153846 ... 0.20560748 0.09345794 0.        ]
 ...
 [0.48717949 0.45454545 0.40384615 ... 0.03738318 0.06542056 0.        ]
 [0.61538462 0.48181818 0.63461538 ... 0.03738318 0.06542056 0.        ]
 [0.55555556 0.51818182 0.42307692 ... 0.09345794 0.03738318 0.        ]]


**TODO:** Can add more columns from the dataset. 

- `TARGET_COL` — binary outcome (`unsafe_weather`).
- `WIND_FEATURES`, `RAIN_FEATURES` — join columns together to combine all features



In [ ]:
print("Columns:", list(df.columns))

# --- STUDENT: replace with your dataset's column names ---
TARGET_COL = "unsafe_weather"

WIND_FEATURES = ["avg_wind_speed_kts"] 
 # TODO: Fill out the rest of them 
RAIN_FEATURES = ["precip_inches", "humidity_pct"]
TEMP_FEATURES = ["lightning_strikes_10mi", "wind_gust"]

# can add features for snow later 

MODEL_PLANS = {
    "wind_knn": {"features": WIND_FEATURES, "target": TARGET_COL},
    "rain_knn": {"features": RAIN_FEATURES, "target": TARGET_COL},
    "temp_knn": {"features": TEMP_FEATURES, "target": TARGET_COL},
}



## 2. Helpers — implement these

Complete the bodies of the functions below. Keep functions small and test them on a few rows before training.

In [ ]:
def extract_xy(df: pd.DataFrame, feature_cols: list[str], target_col: str):
    """Return X (features) and y (target) as numpy arrays suitable for sklearn.

    TODO:
    - Select columns; drop rows with missing values in those columns (or impute — your choice).
    - Cast target to integer 0/1 (handle string '0'/'1' if present).
    """
    raise NotImplementedError("Student: implement extract_xy")


def scale_train_test(X_train, X_test):
    """Fit StandardScaler on X_train only; return scaler, X_train_scaled, X_test_scaled."""
    raise NotImplementedError("Student: implement scale_train_test")


def train_knn(X_train, y_train, n_neighbors: int = 5):
    """Create and fit KNeighborsClassifier. """
    raise NotImplementedError("Student: implement train_knn")


def evaluate(model, X_test, y_test, label: str = "model"):
    """Print accuracy, confusion matrix, and classification report."""
    raise NotImplementedError("Student: implement evaluate")

## 3. Train and compare the three KNNs

**TODO:** After implementing the helpers, loop over `MODEL_PLANS`. Use the same test set size and `random_state` for fair comparison.

Can start with: `test_size=0.25`, `n_neighbors=5`, `random_state=42`.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.25
N_NEIGHBORS = 5

# after implementing helpers, run training for each entry in MODEL_PLANS ---
# Pseudocode:
# for name, spec in MODEL_PLANS.items():
#     X, y = extract_xy(df, spec["features"], spec["target"])
#     X_train, X_test, y_train, y_test = train_test_split(...)
#     scaler, X_train_s, X_test_s = scale_train_test(X_train, X_test)
#     model = train_knn(X_train_s, y_train, n_neighbors=N_NEIGHBORS)
#     evaluate(model, X_test_s, y_test, label=name)

raise NotImplementedError("wire up the training loop using your helper functions")

## 4. Extension — sweep \(k\)

**TODO:** For your best-performing feature group, plot or print test accuracy for `n_neighbors` in `{1, 3, 5, 7, 9, 15}`.

In [ ]:
# use matplot lib to show these 
# import matplotlib.pyplot as plt
# ks = [1, 3, 5, 7, 9, 15]
# scores = []
# ...
# plt.plot(ks, scores, marker="o")
pass